# nnU-Net BraTS 2024 MEN-RT — Full Pipeline (Kaggle)

**Before running:**
1. GPU must be ON → Notebook Settings → Accelerator → **GPU T4 x2** or **P100**
2. Add your BraTS dataset as a Kaggle Dataset input
3. Add your GitHub token as a Kaggle Secret named `nnunet_kaggle`

**Pipeline overview:**
- Steps 1–2: Data preparation and nnU-Net preprocessing
- Step 3: K-fold cross-validation training (fold-by-fold)
- Step 4: Per-fold inference → segmentation predictions
- Steps 5–6: Evaluation and visualization

**Resuming an interrupted session:**
Re-run Cells 1–7, set `CONTINUE_TRAINING = True` in the relevant training cell, then re-run that cell.

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess, os, sys

try:
    smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(smi.stdout)
except FileNotFoundError:
    print('nvidia-smi not found — GPU is not enabled yet.\n')
    print('ACTION REQUIRED:')
    print('  1. Right side panel → Settings (pencil icon)')
    print('  2. Under Accelerator → select GPU T4 x2 or P100')
    print('  3. Click Save → kernel restarts automatically')
    print('  4. Re-run this cell — it should show GPU info')

import torch
cuda_ok = torch.cuda.is_available()
print(f'CUDA available : {cuda_ok}')
if cuda_ok:
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name} | {p.total_memory / 1e9:.1f} GB')
    print('\nGPU OK — safe to continue to Cell 2.')
else:
    raise RuntimeError(
        '\nNo GPU detected!\n'
        'Go to: right panel → Settings → Accelerator → GPU T4 x2\n'
        'Save, wait for kernel restart, then re-run this cell.'
    )

In [ ]:
# ── Cell 2: Locate BraTS dataset from Kaggle input ────────────────────────────
import glob, re
from collections import Counter
from pathlib import Path

KAGGLE_INPUT = '/kaggle/input'

# ── Known exact extracted paths (confirmed working structure) ─────────────────
_KNOWN_TRAIN = (
    f'{KAGGLE_INPUT}/datasets/maryampervaiz24029/brats-men-rt'
    '/BraTS2024-MEN-RT-TrainingData/BraTS-MEN-RT-Train-v2'
)
_KNOWN_VAL = (
    f'{KAGGLE_INPUT}/datasets/maryampervaiz24029/brats-men-rt'
    '/BraTS2024-MEN-RT-ValidationData/BraTS-MEN-RT-Val-v1'
)

def _find_zip(root: str, *names: str) -> str | None:
    for name in names:
        hits = sorted(glob.glob(f'{root}/**/{name}', recursive=True))
        if hits:
            return hits[0]
    return None

def _find_case_root(root: str) -> str | None:
    for pat in ('*_t1c.nii.gz', '*_t1c.nii', '*_t1ce.nii.gz', '*_t1ce.nii',
                '*_t1n.nii.gz', '*_t1n.nii', '*_t1.nii.gz', '*_t1.nii'):
        hits = sorted(glob.glob(f'{root}/**/{pat}', recursive=True))
        if hits:
            return str(Path(hits[0]).parent.parent)
    return None

def _scan_suffixes(root: str, max_files: int = 300) -> Counter:
    c: Counter = Counter()
    rex = re.compile(r'^.+?_([^_]+)\.nii(?:\.gz)?$', re.IGNORECASE)
    for pat in ('**/*.nii.gz', '**/*.nii'):
        for f in glob.glob(f'{root}/{pat}', recursive=True)[:max_files]:
            m = rex.match(Path(f).name)
            if m:
                c[m.group(1).lower()] += 1
    return c

# ── Priority 1: known extracted paths ────────────────────────────────────────
TRAIN_ZIP = _KNOWN_TRAIN if Path(_KNOWN_TRAIN).is_dir() else None
VAL_ZIP   = _KNOWN_VAL   if Path(_KNOWN_VAL).is_dir()   else None

# ── Priority 2: zip files ─────────────────────────────────────────────────────
if TRAIN_ZIP is None:
    TRAIN_ZIP = _find_zip(KAGGLE_INPUT,
        'BraTS2024-MEN-RT-TrainingData.zip', 'BraTS-MEN-RT-50cases.zip')
if VAL_ZIP is None:
    VAL_ZIP = _find_zip(KAGGLE_INPUT, 'BraTS2024-MEN-RT-ValidationData.zip')

# ── Priority 3: generic discovery ────────────────────────────────────────────
if TRAIN_ZIP is None:
    TRAIN_ZIP = _find_case_root(KAGGLE_INPUT)

print('Dataset discovery results:')
print(f'  Training source : {TRAIN_ZIP}')
print(f'  Validation src  : {VAL_ZIP}')

# ── Diagnostic ────────────────────────────────────────────────────────────────
suffix_counts: Counter = Counter()
if TRAIN_ZIP and not TRAIN_ZIP.endswith('.zip'):
    suffix_counts = _scan_suffixes(TRAIN_ZIP)
    print(f'\nNIfTI suffixes in training root:')
    for suf, n in sorted(suffix_counts.items(), key=lambda x: -x[1]):
        tag = ' <- IMAGE' if suf in ('t1c','t1ce','t1n','t1','flair') else (' <- LABEL' if suf == 'gtv' else '')
        print(f'  _{suf} : {n:4d} files{tag}')
    n_cases = sum(1 for p in Path(TRAIN_ZIP).iterdir() if p.is_dir())
    print(f'  Case folders : {n_cases}')
elif TRAIN_ZIP and TRAIN_ZIP.endswith('.zip'):
    print('  Source is a zip — extracted during conversion.')

# ── Validate ──────────────────────────────────────────────────────────────────
if TRAIN_ZIP is None:
    raise FileNotFoundError(
        'BraTS training data not found.\n'
        'Add the dataset: right panel -> Add Input -> Your Datasets -> brats-men-rt.'
    )

image_ok = any(s in suffix_counts for s in ('t1c','t1ce','t1n','t1','flair'))
if not TRAIN_ZIP.endswith('.zip') and suffix_counts and not image_ok:
    raise FileNotFoundError(
        f'No T1c image files found in {TRAIN_ZIP}\n'
        'Each case folder needs *_t1c.nii or *_t1c.nii.gz + *_gtv.nii or *_gtv.nii.gz'
    )

print('\nDataset found. Ready to proceed.')

In [ ]:
# ── Cell 3: Clone repository from GitHub ─────────────────────────────────────
import os, subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

# Ensure working directory is valid before any git operations
os.makedirs('/kaggle/working', exist_ok=True)
os.chdir('/kaggle/working')

secrets  = UserSecretsClient()
token    = secrets.get_secret('nnunet_kaggle')
repo_url = f'https://{token}@github.com/maryampervaiz-alt/nnunet-training.git'
REPO_DIR = '/kaggle/working/nnunet-training'

subprocess.run(['rm', '-rf', REPO_DIR], capture_output=True)
result = subprocess.run(
    ['git', 'clone', '--depth', '1', repo_url, REPO_DIR],
    capture_output=True, text=True,
)
if result.returncode != 0:
    raise RuntimeError(
        f'git clone failed (rc={result.returncode}):\n{result.stderr}\n'
        'Check that your nnunet_kaggle secret contains a valid GitHub token.'
    )

print('Repository cloned successfully.')
print('Contents:')
for item in sorted(Path(REPO_DIR).iterdir()):
    print(f'  {item.name}/' if item.is_dir() else f'  {item.name}')


In [ ]:
# ── Cell 4: Install dependencies ─────────────────────────────────────────────
import subprocess, sys

def pip_install(*packages: str, quiet: bool = True) -> None:
    """Install one or more packages via pip."""
    args = [sys.executable, '-m', 'pip', 'install'] + list(packages)
    if quiet:
        args.append('-q')
    result = subprocess.run(args)
    if result.returncode != 0:
        raise RuntimeError(f'pip install failed for: {packages}')

print('Installing surface-distance (from GitHub) ...')
pip_install('git+https://github.com/google-deepmind/surface-distance.git')

print('Installing nnunetv2 ...')
pip_install('nnunetv2')

print('Installing remaining dependencies ...')
pip_install(
    'nibabel', 'SimpleITK', 'scipy', 'scikit-image',
    'pandas', 'matplotlib', 'seaborn',
    'loguru', 'rich', 'python-dotenv', 'tqdm', 'mlflow',
)

print('\nAll packages installed.')

In [ ]:
# ── Cell 5: Register custom trainer with nnunetv2 (CRITICAL) ─────────────────
# nnU-Net discovers trainer classes by scanning its own package directory.
# Our custom trainer is outside that package, so we copy it in.

import shutil, importlib.util, os
from pathlib import Path

# Set nnunet env vars NOW so nnunetv2 import doesn't print "not defined" warnings
PROJECT = REPO_DIR  # defined in Cell 3
os.environ.setdefault('nnUNet_raw',          f'{PROJECT}/nnunet_raw')
os.environ.setdefault('nnUNet_preprocessed', f'{PROJECT}/nnunet_preprocessed')
os.environ.setdefault('nnUNet_results',      f'{PROJECT}/nnunet_results')

import nnunetv2

nnunet_pkg_dir = Path(nnunetv2.__file__).parent
trainer_src    = Path(REPO_DIR) / 'src' / 'training' / 'nnunet_trainer_es.py'
trainer_dst    = nnunet_pkg_dir / 'training' / 'nnUNetTrainer' / 'nnUNetTrainerEarlyStopping.py'

if not trainer_src.exists():
    raise FileNotFoundError(f'Custom trainer source not found: {trainer_src}')

shutil.copy2(trainer_src, trainer_dst)
print(f'Registered: {trainer_src.name} → {trainer_dst}')

# Verify the class is importable
spec = importlib.util.spec_from_file_location('nnUNetTrainerEarlyStopping', trainer_dst)
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
cls  = getattr(mod, 'nnUNetTrainerEarlyStopping', None)
if cls is None:
    raise ImportError('nnUNetTrainerEarlyStopping not found in copied file')

print(f'Trainer class   : {cls}')
print('Custom trainer registration: OK')

In [ ]:
# ── Cell 6: Configure environment variables ───────────────────────────────────

PROJECT   = REPO_DIR
MAX_CASES = 20     # 20-case subset for quick pilot run

os.environ.update({
    'nnUNet_raw'          : f'{PROJECT}/nnunet_raw',
    'nnUNet_preprocessed' : f'{PROJECT}/nnunet_preprocessed',
    'nnUNet_results'      : f'{PROJECT}/nnunet_results',
    'DATASET_ID'          : '001',
    'DATASET_NAME'        : 'BraTSMENRT',
    'NNUNET_CONFIGURATION': '3d_fullres',
    'NNUNET_SEED'         : '42',
    'ES_PATIENCE'         : '20',
    'ES_MIN_DELTA'        : '0.0001',
    'ES_WARMUP'           : '20',
    'NUM_FOLDS'           : '5',
    'CUDA_VISIBLE_DEVICES': '0',
    'TRAIN_SOURCE'        : str(TRAIN_ZIP),
    'VAL_SOURCE'          : str(VAL_ZIP) if VAL_ZIP else '',
    'LABEL_SUFFIX'        : 'gtv',
    'EXPERIMENT_NAME'     : 'BraTSMENRT_baseline',
    'MLFLOW_TRACKING_URI' : f'{PROJECT}/experiments/mlruns',
    # Disable Python stdout buffering so subprocess output appears immediately.
    'PYTHONUNBUFFERED'    : '1',
    # Limit DA workers to avoid fork-based deadlocks inside Kaggle subprocess chain.
    'nnUNet_n_proc_DA'    : '4',
    # Disable torch.compile to avoid 20-40 min JIT compilation on the first epoch.
    # On the full dataset with many epochs, set this to 'true' for faster training.
    'nnUNet_compile'      : 'false',
})

for d in [
    'nnunet_raw', 'nnunet_preprocessed', 'nnunet_results',
    'metrics', 'results', 'visualizations',
    'inference_outputs', 'logs', 'experiments', 'checkpoints',
]:
    Path(f'{PROJECT}/{d}').mkdir(parents=True, exist_ok=True)

print('Environment configured:')
for k in ['nnUNet_raw', 'nnUNet_preprocessed', 'nnUNet_results',
          'DATASET_ID', 'DATASET_NAME', 'NNUNET_CONFIGURATION']:
    print(f'  {k:25} = {os.environ[k]}')
print(f'  TRAIN_SOURCE              = {TRAIN_ZIP}')
print(f'  MAX_CASES                 = {MAX_CASES}')
print(f'  torch.compile             = {os.environ.get("nnUNet_compile", "true")}')


In [ ]:
# ── Cell 7: Set working directory to the repo ─────────────────────────────────
# All subsequent ! commands and relative paths resolve from here.

os.chdir(PROJECT)

# Add repo root to Python path so src.* imports work in notebook cells
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

print(f'Working directory: {os.getcwd()}')
print(f'Python path includes: {sys.path[0]}')

In [ ]:
# ── Cell 8: STEP 1 — Convert raw BraTS data to nnU-Net format ────────────
# Expected time: 2-4 minutes for 20 cases (training only, validation skipped).

cmd = [
    sys.executable, '-u', 'scripts/01_prepare_dataset.py',
    '--train',    str(TRAIN_ZIP),
    '--skip-val',
    '--log-dir',  'logs',
]
if MAX_CASES is not None:
    cmd += ['--max-cases', str(MAX_CASES)]

cmd_str = ' '.join(cmd)
print(f'Running: {cmd_str}')
print()

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=os.environ.copy(),
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'01_prepare_dataset.py failed (rc={proc.returncode})')
print()
print('Dataset preparation complete.')


In [ ]:
# ── Cell 9: Verify dataset structure ─────────────────────────────────────────
import json
from pathlib import Path

raw_dir     = Path(os.environ['nnUNet_raw'])
dataset_dir = raw_dir / 'Dataset001_BraTSMENRT'
images_tr   = dataset_dir / 'imagesTr'
labels_tr   = dataset_dir / 'labelsTr'
images_ts   = dataset_dir / 'imagesTs'

n_tr  = len(list(images_tr.glob('*_0000.nii.gz')))
n_lbl = len(list(labels_tr.glob('*.nii.gz')))
n_ts  = len(list(images_ts.glob('*_0000.nii.gz'))) if images_ts.exists() else 0

with open(dataset_dir / 'dataset.json') as fh:
    ds = json.load(fh)

print(f'imagesTr images  : {n_tr}')
print(f'labelsTr labels  : {n_lbl}')
print(f'imagesTs images  : {n_ts}')
print(f'numTraining (json): {ds["numTraining"]}')
print(f'channel_names    : {ds["channel_names"]}')
print(f'labels           : {ds["labels"]}')

assert n_tr == n_lbl, f'Image/label count mismatch: {n_tr} vs {n_lbl}'
assert ds['numTraining'] == n_tr, (
    f'numTraining in dataset.json ({ds["numTraining"]}) does not match '
    f'actual file count ({n_tr})'
)
print('\nDataset structure OK.')

In [ ]:
# ── Cell 10: STEP 2 — nnU-Net Planning & Preprocessing ───────────────────────
# Runs nnUNetv2_plan_and_preprocess. Runtime scales with dataset size:
# ~5 min for 20 cases, ~20-40 min for the full dataset.

proc = subprocess.Popen(
    [sys.executable, '-u', 'scripts/02_preprocess.py', '--log-dir', 'logs'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=os.environ.copy(),
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'02_preprocess.py failed (rc={proc.returncode})')
print('Preprocessing complete.')


In [ ]:
# ── Cell 11: Integrity Check — Validate Dataset Before Training ───────────────
# Checks every case for:
#   - Readable NIfTI files
#   - Matching image/label spatial dimensions
#   - Binary label values {0, 1}
#   - No duplicate case IDs

from src.data.integrity_checker import IntegrityChecker

checker = IntegrityChecker(dataset_dir=dataset_dir, expected_label_values={0, 1})
report  = checker.run()

print('=' * 60)
print('  INTEGRITY CHECK REPORT')
print('=' * 60)
print(report.summary())
print('=' * 60)

failed = [r for r in report.case_reports if not r.ok]
if failed:
    for r in failed:
        print(f'  FAIL [{r.case_id}]: {r.errors}')
    raise AssertionError(
        f'{len(failed)} case(s) failed integrity checks. '
        'Fix the issues above before proceeding.'
    )

print('All cases passed integrity checks. Safe to train.')


In [ ]:
# ── Cell 12: Verify preprocessing + show CV splits ───────────────────────────

preproc_dir = Path(os.environ['nnUNet_preprocessed']) / 'Dataset001_BraTSMENRT'

# List preprocessed files
print('Preprocessed directory contents:')
for item in sorted(preproc_dir.iterdir()):
    print(f'  {item.name}')

# Show splits
splits_file = preproc_dir / 'splits_final.json'
if splits_file.exists():
    with open(splits_file) as fh:
        splits = json.load(fh)
    print(f'\n5-fold cross-validation splits ({len(splits)} folds):')
    for i, fold in enumerate(splits):
        print(f'  Fold {i}: {len(fold["train"])} train | {len(fold["val"])} val')
else:
    print('\nsplits_final.json not found — preprocessing may not have completed.')

# Show nnU-Net plans
for plans_file in sorted(preproc_dir.glob('nnUNetPlans*.json')):
    with open(plans_file) as fh:
        plans = json.load(fh)
    print(f'\nPlans from {plans_file.name}:')
    for cfg_name, cfg in plans.get('configurations', {}).items():
        print(
            f'  [{cfg_name}]  '
            f'patch_size={cfg.get("patch_size")}  '
            f'spacing={cfg.get("spacing")}  '
            f'batch_size={cfg.get("batch_size")}'
        )

In [ ]:
# ── Cell 13: STEP 3 — Train Fold 0 ───────────────────────────────────────────
# Streams epoch-level logs in real time.
# Runtime: ~3-5 min/epoch on T4 GPU.

FOLD             = 0
NUM_EPOCHS       = 50
CONTINUE_TRAINING = False   # set True to resume from checkpoint_latest.pth

cmd = [
    sys.executable, '-u', 'scripts/03_train.py',
    '--folds',           str(FOLD),
    '--num-epochs',      str(NUM_EPOCHS),
    '--seed',            '42',
    '--trainer',         'nnUNetTrainerEarlyStopping',
    '--es-patience',     '50',
    '--es-min-delta',    '0.0001',
    '--es-warmup',       '50',
    '--log-dir',         'logs',
    '--metrics-dir',     'metrics',
    '--checkpoints-dir', 'checkpoints',
]
if CONTINUE_TRAINING:
    cmd.append('--continue-training')

cmd_str = ' '.join(cmd)
print(f'Training fold {FOLD} for {NUM_EPOCHS} epochs '
      f'({"resuming" if CONTINUE_TRAINING else "fresh start"}) ...')
print(f'Command: {cmd_str}')
print()

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=os.environ.copy(),
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode != 0:
    raise RuntimeError(f'Fold {FOLD} training failed (rc={proc.returncode})')

print()
print(f'Fold {FOLD} training complete.')
print(f'  Native checkpoints : {os.environ["nnUNet_results"]}/')
print(f'  Training log       : metrics/fold_{FOLD}_training.csv')


In [ ]:
# ── Cell 13b: Training History — Per-epoch metrics after fold 0 training ────────
# Reads the CSV written during training and prints a concise summary table.

import pandas as pd

hist_csv = Path('metrics/fold_0_training.csv')
if not hist_csv.exists():
    print(f'Training history not found: {hist_csv}')
    print('Run Cell 13 first.')
else:
    hist = pd.read_csv(hist_csv)
    print(f'Fold 0 training history: {len(hist)} epochs')
    if 'val_dice' in hist.columns and not hist['val_dice'].isna().all():
        best_ep = int(hist['val_dice'].idxmax())
        best_dsc = hist['val_dice'].max()
        final_dsc = hist['val_dice'].iloc[-1]
        print(f'  Best val Dice   : {best_dsc:.4f}  (epoch {best_ep})')
        print(f'  Final val Dice  : {final_dsc:.4f}')
    if 'train_loss' in hist.columns:
        print(f'  Final train loss: {hist["train_loss"].iloc[-1]:.4f}')
    print()
    display_cols = [c for c in ['epoch', 'train_loss', 'val_loss', 'val_dice',
                                'epoch_time_s'] if c in hist.columns]
    step = max(1, len(hist) // 10)
    print('Epoch log (every ~10th epoch):')
    print(hist.iloc[::step][display_cols].round(4).to_string(index=False))
    print()
    print(f'Full training log: {hist_csv}')


In [ ]:
# ── Cell 14: Verify Fold 0 Checkpoints ───────────────────────────────────────
# Confirms that nnU-Net saved checkpoint_best.pth and checkpoint_latest.pth.

TRAINER = 'nnUNetTrainerEarlyStopping'
PLANS   = 'nnUNetPlans'
CONFIG  = os.environ['NNUNET_CONFIGURATION']

nnunet_ckpt_dir = (
    Path(os.environ['nnUNet_results'])
    / f'Dataset{os.environ["DATASET_ID"].zfill(3)}_{os.environ["DATASET_NAME"]}'
    / f'{TRAINER}__{PLANS}__{CONFIG}'
    / f'fold_{FOLD}'
)

print(f'Checkpoint directory: {nnunet_ckpt_dir}')
print()

for fname in ['checkpoint_best.pth', 'checkpoint_latest.pth']:
    p = nnunet_ckpt_dir / fname
    if p.exists():
        print(f'  OK       {fname:30} ({p.stat().st_size / 1e6:.1f} MB)')
    else:
        print(f'  MISSING  {fname}')

log_files = sorted(nnunet_ckpt_dir.glob('training_log*.txt'))
if log_files:
    print(f'\nLast 10 lines of training log ({log_files[-1].name}):')
    lines = log_files[-1].read_text().splitlines()
    print('\n'.join(lines[-10:]))

print('\nCheckpoint verification complete.')


---
## After Fold 0 Training

Proceed in order:
- **Cell 14** → Verify that `checkpoint_best.pth` and `checkpoint_latest.pth` exist
- **Cell 15** → Run fold-0 inference → `inference_outputs/cv/fold_0/*.nii.gz`
- **Cell 15b** → Post-processing (connected component filtering)
- **Cell 16** → Evaluate metrics (DSC, HD95, NSD, Precision, Recall)
- **Cell 17** → Visualizations
- **Cell 18** → Full metrics table
- **Cell 20** → Save all outputs before the session ends
- **Cell 21** → Train fold 1 (same configuration)

In [ ]:
# ── Cell 15: STEP 4 — Inference using Fold 0 model ───────────────────────
# Predicts GTV masks on the held-out validation cases of fold 0.
# Output: inference_outputs/cv/fold_0/*.nii.gz
# Expected time: 10-30 minutes

proc = subprocess.Popen(
    [
        sys.executable, '-u', 'scripts/04_inference.py',
        '--cv-mode',
        '--folds',    '0',
        '--output',   'inference_outputs/cv',
        '--log-dir',  'logs',
    ],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=os.environ.copy(),
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode != 0:
    raise RuntimeError(f'04_inference.py failed (rc={proc.returncode})')

pred_dir = Path('inference_outputs/cv/fold_0')
preds    = list(pred_dir.glob('*.nii.gz')) if pred_dir.exists() else []
print(f'Inference complete. {len(preds)} predictions in {pred_dir}')


In [ ]:
# ── Cell 15b: Post-processing — Connected Component Filtering ──────────────
# Removes small disconnected foreground components (likely false positives).
# Compares DSC / HD95 / NSD before and after to confirm whether it helps.

import numpy as np
import nibabel as nib
from scipy import ndimage
import pandas as pd

MIN_VOXELS = 50


def postprocess_mask(arr, min_voxels=MIN_VOXELS):
    labeled, n_comp = ndimage.label(arr > 0)
    if n_comp == 0:
        return arr.copy().astype(np.uint8)
    sizes = ndimage.sum(arr > 0, labeled, range(1, n_comp + 1))
    keep = np.zeros_like(labeled, dtype=bool)
    for idx, sz in enumerate(sizes, start=1):
        if sz >= min_voxels:
            keep |= (labeled == idx)
    return keep.astype(np.uint8)


pred_dir     = Path('inference_outputs/cv/fold_0')
postproc_dir = Path('inference_outputs/cv/fold_0_postproc')
postproc_dir.mkdir(parents=True, exist_ok=True)

preds = sorted(pred_dir.glob('*.nii.gz')) if pred_dir.exists() else []
if not preds:
    print('No predictions found. Run Cell 15 first.')
else:
    print(f'Post-processing {len(preds)} predictions (min_voxels={MIN_VOXELS}) ...')
    n_changed = 0
    for pred_path in preds:
        img     = nib.load(pred_path)
        arr     = np.asarray(img.dataobj, dtype=np.uint8)
        cleaned = postprocess_mask(arr)
        if not np.array_equal(arr, cleaned):
            n_changed += 1
        nib.save(nib.Nifti1Image(cleaned, img.affine, img.header),
                 postproc_dir / pred_path.name)

    print(f'  {n_changed}/{len(preds)} predictions modified.')
    print(f'  Saved to: {postproc_dir}')
    print()

    gt_dir = (Path(os.environ['nnUNet_raw'])
              / 'Dataset001_BraTSMENRT' / 'labelsTr')

    proc = subprocess.Popen(
        [
            sys.executable, '-u', 'scripts/05_evaluate.py',
            '--pred',        str(postproc_dir),
            '--gt',          str(gt_dir),
            '--results-dir', 'results',
            '--tag',         'fold_0_postproc',
            '--log-dir',     'logs',
        ],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=os.environ.copy(),
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()

    pre_csv  = Path('results/fold_0_per_case.csv')
    post_csv = Path('results/fold_0_postproc_per_case.csv')
    if pre_csv.exists() and post_csv.exists():
        pre  = pd.read_csv(pre_csv)
        post = pd.read_csv(post_csv)
        print()
        print('Metric comparison: raw vs. post-processed predictions')
        print(f'  {"Metric":<25} {"Before":>10} {"After":>10} {"Delta":>10}')
        print('  ' + '-' * 57)
        for col in ['dice', 'hd95', 'nsd', 'precision', 'recall']:
            if col not in pre.columns or col not in post.columns:
                continue
            b = pre[col].replace([float('inf')], float('nan')).mean()
            a = post[col].replace([float('inf')], float('nan')).mean()
            d = a - b
            sign = '+' if d > 0 else ''
            print(f'  {col:<25} {b:>10.4f} {a:>10.4f} {sign}{d:>9.4f}')


In [ ]:
# ── Cell 16: STEP 5 — Evaluate Fold 0 ────────────────────────────────────
# Computes: DSC, HD95, NSD (2 mm), Precision, Recall, Specificity, Volume Error
# Expected time: 5-15 minutes

proc = subprocess.Popen(
    [
        sys.executable, '-u', 'scripts/05_evaluate.py',
        '--cv-mode',
        '--results-dir',  'results',
        '--show-rankings',
        '--log-dir',      'logs',
    ],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=os.environ.copy(),
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode != 0:
    raise RuntimeError(f'05_evaluate.py failed (rc={proc.returncode})')

print('Evaluation complete.')


In [ ]:
# ── Cell 17: STEP 6 — Visualizations ─────────────────────────────────────────
# Generates: segmentation overlays, violin/box plots, training curves
# Expected time: 2–5 minutes

result = subprocess.run(
    [
        sys.executable, 'scripts/06_visualize.py',
        '--all',
        '--cv-mode',
        '--results-dir',  'results',
        '--metrics-dir',  'metrics',
        '--output-dir',   'visualizations',
        '--log-dir',      'logs',
    ],
    env=os.environ.copy(),
)
if result.returncode != 0:
    print(f'Warning: 06_visualize.py exited with rc={result.returncode} '
          '(non-fatal — some plots may be missing)')

In [ ]:
# -- Cell 18: Quantitative Results -- Full Metrics Table
# All segmentation metrics in a clean table for thesis/publication reporting.

import pandas as pd
import numpy as np

_METRIC_DISPLAY = {
    "dice":                "DSC",
    "hd95":                "HD95 (mm)",
    "hd":                  "HD (mm)",
    "nsd":                 "NSD",
    "precision":           "Precision",
    "recall":              "Recall (Sensitivity)",
    "specificity":         "Specificity",
    "volume_similarity":   "Volume Similarity",
    "abs_volume_error_ml": "Abs. Volume Error (ml)",
}


def _show_metrics_table(csv_path, title):
    p = Path(csv_path)
    if not p.exists():
        print(f"{title}: file not found at {p}")
        return None
    df = pd.read_csv(p)
    metric_cols = [c for c in _METRIC_DISPLAY if c in df.columns]
    if not metric_cols:
        print(f"{title}: no recognised metric columns")
        return None

    rows = []
    for col in metric_cols:
        vals = pd.to_numeric(df[col], errors="coerce")
        vals = vals.replace([float("inf"), float("-inf")], float("nan")).dropna()
        if vals.empty:
            continue
        rows.append({
            "Metric":          _METRIC_DISPLAY[col],
            "Mean +/- Std":    f"{vals.mean():.4f} +/- {vals.std():.4f}",
            "Median [IQR]":    (
                f"{vals.median():.4f} "
                f"[{vals.quantile(0.25):.4f}, {vals.quantile(0.75):.4f}]"
            ),
            "Min":             f"{vals.min():.4f}",
            "Max":             f"{vals.max():.4f}",
            "N":               int(vals.count()),
        })

    summary = pd.DataFrame(rows).set_index("Metric")
    print(f"\n{title} ({len(df)} cases)")
    print("=" * 95)
    print(summary.to_string())
    print("=" * 95)

    if "dice" in df.columns:
        n_out = int((df["dice"] < 0.5).sum())
        if n_out:
            print(f"  Outliers (DSC < 0.5): {n_out} cases")
            if "case_id" in df.columns:
                low = df[df["dice"] < 0.5][["case_id", "dice"]].round(4)
                print(low.to_string(index=False))
    return summary


_show_metrics_table("results/fold_0_per_case.csv",  "Fold 0 Validation Results")
_show_metrics_table("results/cv_combined.csv",       "Cross-Validation Combined Results")


In [ ]:
# ── Cell 19: Show overlay images and metric plots ─────────────────────────────

from IPython.display import Image, display

# Show up to 3 segmentation overlays
overlay_dir = Path('visualizations/overlays')
if overlay_dir.exists():
    for img_path in sorted(overlay_dir.glob('*.png'))[:3]:
        print(f'\n{img_path.name}')
        display(Image(str(img_path), width=900))
else:
    print('No overlay images found — run Cell 17 first')

# Show metric plots
for plot in [
    'visualizations/metrics_violin.png',
    'visualizations/metrics_boxplot.png',
    'visualizations/volume_scatter.png',
    'visualizations/training_curves.png',
]:
    if Path(plot).exists():
        print(f'\n{plot}')
        display(Image(plot, width=900))

In [ ]:
# ── Cell 20: Save all outputs to /kaggle/working ──────────────────────────────
# Kaggle deletes /kaggle/working/nnunet-training when the session ends.
# This cell copies everything important to /kaggle/working/outputs which IS
# preserved as a notebook output and can be downloaded.

import shutil

SAVE = '/kaggle/working/outputs'

to_save = [
    ('results',             f'{SAVE}/results'),
    ('metrics',             f'{SAVE}/metrics'),
    ('visualizations',      f'{SAVE}/visualizations'),
    ('inference_outputs',   f'{SAVE}/inference_outputs'),
    ('logs',                f'{SAVE}/logs'),
    ('checkpoints',         f'{SAVE}/checkpoints'),
    # Also save the native nnU-Net checkpoints (checkpoint_best.pth, etc.)
    (os.environ['nnUNet_results'], f'{SAVE}/nnunet_results'),
]

Path(SAVE).mkdir(parents=True, exist_ok=True)

for src, dst in to_save:
    if Path(src).exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        # Count files saved
        n = sum(1 for _ in Path(dst).rglob('*') if Path(_).is_file())
        print(f'Saved {n:4d} files: {src} → {dst}')
    else:
        print(f'Skipped (not found): {src}')

print(f'\nAll outputs saved to {SAVE}')

# Summary of checkpoint files
print('\nCheckpoint files:')
for p in sorted(Path(SAVE).rglob('*.pth')):
    print(f'  {p.relative_to(SAVE)} ({p.stat().st_size / 1e6:.1f} MB)')

---
## Train Fold 1 (Cell 21)

Run Cell 21 after fold-0 inference and evaluation are complete.

**In a new Kaggle session:** re-run Cells 1–7, then run Cell 21 directly.
Cells 8–12 can be skipped if the preprocessed data was saved via Cell 20.

To resume an interrupted fold-1 run, set `CONTINUE_TRAINING = True` in Cell 21.

In [ ]:
# ── Cell 21: STEP 3 — Train Fold 1 ───────────────────────────────────────────
# Streams epoch-level logs in real time.
# Runtime: ~3-5 min/epoch on T4 GPU.

FOLD              = 1
NUM_EPOCHS        = 50
CONTINUE_TRAINING = False   # set True to resume from checkpoint_latest.pth

cmd = [
    sys.executable, '-u', 'scripts/03_train.py',
    '--folds',           str(FOLD),
    '--num-epochs',      str(NUM_EPOCHS),
    '--seed',            '42',
    '--trainer',         'nnUNetTrainerEarlyStopping',
    '--es-patience',     '50',
    '--es-min-delta',    '0.0001',
    '--es-warmup',       '50',
    '--log-dir',         'logs',
    '--metrics-dir',     'metrics',
    '--checkpoints-dir', 'checkpoints',
]
if CONTINUE_TRAINING:
    cmd.append('--continue-training')

cmd_str = ' '.join(cmd)
print(f'Training fold {FOLD} for {NUM_EPOCHS} epochs '
      f'({"resuming" if CONTINUE_TRAINING else "fresh start"}) ...')
print(f'Command: {cmd_str}')
print()

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=os.environ.copy(),
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode != 0:
    raise RuntimeError(f'Fold {FOLD} training failed (rc={proc.returncode})')

print()
print(f'Fold {FOLD} training complete.')
print(f'  Native checkpoints : {os.environ["nnUNet_results"]}/')
print(f'  Training log       : metrics/fold_{FOLD}_training.csv')
print()
print('Next: update FOLD = 1 in Cell 15 and Cell 16, then run inference and evaluation.')
